In [3]:
import os
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage
from langchain_community.chat_models import ChatOllama
from dotenv import load_dotenv
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_upstage import UpstageEmbeddings
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub
from langchain_community.embeddings import OllamaEmbeddings

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  
    chunk_overlap=200,) 

loader = Docx2txtLoader('./tax.docx')

document_list = loader.load_and_split(text_splitter=text_splitter)


#embedding = UpstageEmbeddings(model="solar-embedding-1-large")
#index_name = 'tax-index'
# database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)
#database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory="./chroma")
#retriever = database.as_retriever(search_kwargs={'k': 4})
#retriever.invoke(query)
# retrieved_docs = database.similarity_search(query, k=3)
# query = '연봉 5천만원인 거주자의 종합소득세는?'

embedding = OllamaEmbeddings(model="nomic-embed-text")
database = Chroma.from_documents(documents=document_list, collection_name='chroma-tax_test', embedding=embedding, persist_directory="./chroma")

llm = ChatOllama(model="llama3") 
rag_prompt = hub.pull("rlm/rag-prompt")
retriever = database.as_retriever(search_kwargs={'k': 4})

qa_chain= RetrievalQA.from_chain_type(llm,retriever=retriever, chain_type_kwargs={"prompt":rag_prompt})
dictionary = ["사람을 나타내는 표현 -> 거주자"]

rewrite_prompt = ChatPromptTemplate.from_template("""
사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
그런 경우에는 질문만 리턴해주세요.

사전: {dictionary}

질문: {question}
""")

dictionary_chain = rewrite_prompt | llm | StrOutputParser()
tax_chain = {"query": dictionary_chain} | qa_chain
ai_message = tax_chain.invoke({"question":user_message})

new_question = dictionary_chain.invoke({
    "dictionary": dictionary,
    "question": query
})

result = qa_chain.invoke({"query": new_question})
print(result)


{'query': 'Based on the dictionary, I can rephrase the question to make it more precise and clear. Here is the revised question:\n\n연봉 5천만원인 사람의 종합소득세는?\n\nNote that I replaced "거주자" with "사람", which is a more direct translation of the phrase "사람을 나타내는 표현 -> 거주자". This change makes it clearer that the question is asking about an individual\'s comprehensive income tax, rather than a general description of someone who lives somewhere.', 'result': "Based on the provided context, I can answer your revised question:\n\n연봉 5천만원인 사람의 종합소득세는?\n\nThe comprehensive income tax for an individual with a monthly salary of ₩50 million is not explicitly stated in the given context. However, considering the relevant laws and regulations mentioned, such as the Income Tax Act, it can be inferred that the individual would need to pay taxes on their income according to the applicable tax rates and exemptions.\n\nIn Korea, individuals are taxed on their income based on a progressive income tax system. For s

In [ ]:
%pip install docx2txt
%pip langchain-upstage==0.7.7

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#db 삭제